# Enhanced Heart Disease Prediction Model

This notebook contains the complete implementation of the Enhanced Heart Disease Prediction Model. Features include:
- **21 input features** (original 13 + 8 new medical features)
- **Ensemble Model** using XGBoost, LightGBM, and CatBoost (if available)
- **Risk Percentage** output (instead of binary classification)
- **SHAP Explainability** for individual patient predictions
- **Disease-Specific Risk Scores** (CAD, Hypertension, Heart Failure, Arrhythmia)
- **Personalized Health Recommendations** (diet, exercise, medical consultation)
- **PDF Report Generation** with ReportLab
- **OCR-Based Report Reading** with EasyOCR / PaddleOCR support


In [ ]:
# Install required packages (uncomment if needed)
# !pip install xgboost lightgbm catboost shap reportlab easyocr pandas scikit-learn numpy matplotlib pdf2image

# Enhanced Heart Disease Prediction Model
Features:
- 21 input features (original 13 + 8 new medical features)
- XGBoost / LightGBM / CatBoost ensemble
- Risk percentage output (instead of binary)
- SHAP explainability
- Disease-specific risk scores (CAD, Hypertension, Heart Failure, Arrhythmia)
- Health recommendations
- Downloadable PDF report generation
- OCR-based report reading (EasyOCR / PaddleOCR)

Requirements:
    pip install xgboost lightgbm catboost shap reportlab easyocr pandas scikit-learn numpy matplotlib
    # For PaddleOCR (optional): pip install paddleocr paddlepaddle


## 1. IMPORTS


In [ ]:
import numpy as np
import pandas as pd
import shap
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("[INFO] CatBoost not installed. Ensemble will use XGBoost + LightGBM only.")


## 2. FEATURE DEFINITIONS


In [ ]:
FEATURE_NAMES = [
    # Original 13 features
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal',
    # New 8 features
    'bmi', 'hdl', 'ldl', 'triglycerides',
    'smoking', 'family_history', 'diabetes_history', 'hba1c'
]

FEATURE_DESCRIPTIONS = {
    'age': 'Age (years)',
    'sex': 'Sex (1=Male, 0=Female)',
    'cp': 'Chest Pain Type (0-3)',
    'trestbps': 'Resting Blood Pressure (mmHg)',
    'chol': 'Serum Cholesterol (mg/dl)',
    'fbs': 'Fasting Blood Sugar > 120mg/dl (1=Yes)',
    'restecg': 'Resting ECG Results (0-2)',
    'thalach': 'Max Heart Rate Achieved',
    'exang': 'Exercise Induced Angina (1=Yes)',
    'oldpeak': 'ST Depression (exercise vs rest)',
    'slope': 'Slope of Peak Exercise ST Segment',
    'ca': 'Number of Major Vessels (0-3)',
    'thal': 'Thalassemia (0-3)',
    'bmi': 'Body Mass Index',
    'hdl': 'HDL Cholesterol (mg/dl)',
    'ldl': 'LDL Cholesterol (mg/dl)',
    'triglycerides': 'Triglycerides (mg/dl)',
    'smoking': 'Smoking Status (1=Yes)',
    'family_history': 'Family History of Heart Disease (1=Yes)',
    'diabetes_history': 'Diabetes History (1=Yes)',
    'hba1c': 'HbA1c (%)'
}

NORMAL_RANGES = {
    'trestbps': (90, 120),
    'chol': (0, 200),
    'bmi': (18.5, 24.9),
    'hdl': (60, 100),     # higher is better
    'ldl': (0, 100),
    'triglycerides': (0, 150),
    'hba1c': (4.0, 5.6),
    'thalach': (100, 200),
    'oldpeak': (0, 1.0),
    'ca': (0, 0),
}


## 3. DATA LOADING & PREPROCESSING


In [ ]:

def load_data(csv_path: str) -> pd.DataFrame:
    """Load and validate the enhanced CSV dataset."""
    df = pd.read_csv(csv_path)
    missing = [f for f in FEATURE_NAMES + ['target'] if f not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in dataset: {missing}")
    print(f"[Data] Loaded {len(df)} rows × {len(df.columns)} columns")
    print(f"[Data] Target distribution:\n{df['target'].value_counts().to_string()}")
    return df


def preprocess(df: pd.DataFrame):
    """Split features/target, handle nulls, scale."""
    df = df.dropna()
    X = df[FEATURE_NAMES].copy()
    y = df['target'].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, X_train, X_test


## 4. MODEL TRAINING — XGBoost + LightGBM + CatBoost Ensemble


In [ ]:

def build_ensemble():
    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )

    lgbm = LGBMClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )

    estimators = [('xgb', xgb), ('lgbm', lgbm)]

    if CATBOOST_AVAILABLE:
        cat = CatBoostClassifier(
            iterations=300,
            depth=5,
            learning_rate=0.05,
            random_seed=42,
            verbose=0
        )
        estimators.append(('cat', cat))

    ensemble = VotingClassifier(estimators=estimators, voting='soft')
    return ensemble


def train_model(X_train, y_train, X_test, y_test):
    """Train ensemble and evaluate."""
    print("\n[Model] Training ensemble (XGBoost + LightGBM" +
          (" + CatBoost)" if CATBOOST_AVAILABLE else ")"))

    model = build_ensemble()
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred  = model.predict(X_test)
    test_proba = model.predict_proba(X_test)[:, 1]

    print(f"[Model] Train Accuracy : {accuracy_score(y_train, train_pred):.4f}")
    print(f"[Model] Test  Accuracy : {accuracy_score(y_test,  test_pred):.4f}")
    print(f"[Model] Test  ROC-AUC  : {roc_auc_score(y_test, test_proba):.4f}")
    print("\n[Model] Classification Report:")
    print(classification_report(y_test, test_pred))

    return model


## 5. RISK PERCENTAGE PREDICTION


In [ ]:

def predict_risk_percentage(model, scaler, input_values: dict) -> float:
    """
    Returns heart disease risk as a percentage (0–100).

    input_values: dict with keys matching FEATURE_NAMES
    """
    row = np.array([input_values[f] for f in FEATURE_NAMES]).reshape(1, -1)
    row_scaled = scaler.transform(row)
    prob = model.predict_proba(row_scaled)[0][1]
    return round(prob * 100, 1)


## 6. DISEASE-SPECIFIC RISK SCORES


In [ ]:

def disease_specific_risks(input_values: dict, base_risk: float) -> dict:
    """
    Estimate risk for 4 specific heart conditions using weighted heuristics
    derived from clinical literature, calibrated to the base model risk.
    """
    v = input_values
    br = base_risk / 100.0  # normalize to [0,1]

    def sigmoid(x): return 1 / (1 + np.exp(-x))

    # Coronary Artery Disease — strongly influenced by LDL, cholesterol, smoking, age
    cad_score = (
        0.04 * (v['age'] - 45) +
        0.005 * (v['ldl'] - 100) +
        0.004 * (v['chol'] - 200) +
        0.6 * v['smoking'] +
        0.4 * v['family_history'] +
        0.015 * (v['trestbps'] - 120) +
        2.0 * br
    )
    cad_risk = round(sigmoid(cad_score) * 100, 1)

    # Hypertension — driven by BP, BMI, salt proxy (trestbps), diabetes
    htn_score = (
        0.04 * (v['age'] - 40) +
        0.04 * (v['bmi'] - 25) +
        0.02 * (v['trestbps'] - 120) +
        0.4 * v['diabetes_history'] +
        0.005 * (v['triglycerides'] - 150) +
        1.5 * br
    )
    htn_risk = round(sigmoid(htn_score) * 100, 1)

    # Heart Failure — driven by reduced exercise capacity, existing CAD, age
    hf_score = (
        0.03 * (v['age'] - 50) +
        0.3 * v['exang'] +
        0.2 * v['diabetes_history'] +
        0.04 * (v['bmi'] - 25) +
        0.1 * v['oldpeak'] +
        1.0 * br
    )
    hf_risk = round(sigmoid(hf_score) * 100, 1)

    # Arrhythmia — ECG changes, thalassemia, electrolyte proxy
    arr_score = (
        0.02 * (v['age'] - 40) +
        0.3 * (v['restecg'] > 0) +
        0.2 * (v['thal'] > 1) +
        0.3 * v['exang'] +
        0.5 * br
    )
    arr_risk = round(sigmoid(arr_score) * 100, 1)

    return {
        'Coronary Artery Disease': cad_risk,
        'Hypertension': htn_risk,
        'Heart Failure': hf_risk,
        'Arrhythmia': arr_risk
    }


## 7. SHAP EXPLAINABILITY


In [ ]:

def explain_with_shap(model, scaler, X_train_raw: pd.DataFrame, input_values: dict):
    """
    Compute SHAP values for a single prediction using the XGBoost sub-model.
    Returns sorted list of (feature, shap_value, direction) tuples.
    """
    # Use the XGBoost estimator from the ensemble for SHAP
    xgb_model = None
    # VotingClassifier stores fitted estimators in .estimators_ as a list
    # and the named estimators in .estimators (unfitted) or .named_estimators_
    if hasattr(model, 'named_estimators_'):
        xgb_model = model.named_estimators_.get('xgb', None)
    elif hasattr(model, 'estimators_'):
        # estimators_ is a list of fitted estimators in the same order
        for i, (name, _) in enumerate(model.estimators):
            if name == 'xgb':
                xgb_model = model.estimators_[i]
                break
    if xgb_model is None:
        return []

    X_train_scaled = scaler.transform(X_train_raw)
    explainer = shap.TreeExplainer(xgb_model)

    row = np.array([input_values[f] for f in FEATURE_NAMES]).reshape(1, -1)
    row_scaled = scaler.transform(row)

    shap_values = explainer.shap_values(row_scaled)
    # For binary classification XGBoost returns array of shape (1, n_features)
    sv = shap_values[0] if shap_values.ndim == 2 else shap_values[0]

    pairs = sorted(
        zip(FEATURE_NAMES, sv),
        key=lambda x: abs(x[1]),
        reverse=True
    )

    result = []
    for feat, val in pairs[:10]:
        direction = '↑' if val > 0 else '↓'
        result.append((feat, round(abs(val) * 100, 1), direction, val))
    return result


## 8. EXPLAIN TOP RISK FACTORS (Simple Rule-Based + SHAP)


In [ ]:

def explain_predictions(input_values: dict, shap_factors: list) -> list:
    """
    Returns a list of strings explaining why the risk is high/low,
    combining SHAP importance with clinical normal ranges.
    """
    explanations = []
    v = input_values

    if shap_factors:
        for feat, pct, direction, raw_val in shap_factors[:5]:
            actual = v[feat]
            desc = FEATURE_DESCRIPTIONS.get(feat, feat)
            norm = NORMAL_RANGES.get(feat)
            flag = ""
            if norm:
                lo, hi = norm
                if feat == 'hdl':
                    flag = " ⚠ Low HDL" if actual < lo else ""
                elif actual > hi:
                    flag = f" ⚠ High (normal <{hi})"
                elif actual < lo:
                    flag = f" ⚠ Low (normal >{lo})"
            explanations.append(
                f"{direction} {desc}: {actual}{flag}  [+{pct}% risk contribution]"
            )
    else:
        # Fallback: simple rule-based flags
        checks = [
            (v['chol'] > 240,         f"✓ High Cholesterol = {v['chol']} mg/dl"),
            (v['ldl'] > 160,          f"✓ High LDL = {v['ldl']} mg/dl"),
            (v['trestbps'] > 140,     f"✓ High Blood Pressure = {v['trestbps']} mmHg"),
            (v['age'] > 55,           f"✓ Age = {v['age']}"),
            (v['smoking'] == 1,       "✓ Active Smoker"),
            (v['diabetes_history'] == 1, "✓ Diabetes History"),
            (v['family_history'] == 1,   "✓ Family History of Heart Disease"),
            (v['bmi'] > 30,           f"✓ Obese BMI = {v['bmi']}"),
            (v['hba1c'] > 6.5,        f"✓ Elevated HbA1c = {v['hba1c']}%"),
            (v['triglycerides'] > 200, f"✓ High Triglycerides = {v['triglycerides']} mg/dl"),
        ]
        for condition, msg in checks:
            if condition:
                explanations.append(msg)

    return explanations if explanations else ["No significant risk factors detected."]


## 9. HEALTH RECOMMENDATIONS


In [ ]:

def generate_recommendations(input_values: dict, risk_pct: float) -> dict:
    """Return personalised diet, exercise and doctor recommendations."""
    v = input_values
    diet, exercise, doctor = [], [], []

    # ── Diet ──
    if v['chol'] > 200 or v['ldl'] > 130:
        diet.append("Reduce saturated fats; choose olive oil, nuts, and avocados.")
        diet.append("Eat fatty fish (salmon, mackerel) 2× per week for Omega-3.")
    if v['triglycerides'] > 150:
        diet.append("Limit refined carbs and sugars; avoid sweetened beverages.")
    if v['bmi'] > 25:
        diet.append("Target a calorie deficit of 300–500 kcal/day for gradual weight loss.")
    if v['hdl'] < 40:
        diet.append("Increase HDL by consuming whole grains, beans, and moderate nuts.")
    if v['hba1c'] > 5.7 or v['diabetes_history']:
        diet.append("Adopt a low glycaemic-index diet; limit white rice and bread.")
    if v['trestbps'] > 130:
        diet.append("Follow a DASH diet; reduce sodium intake to <2,300 mg/day.")
    diet.append("Eat ≥5 servings of vegetables and fruits daily.")
    diet.append("Stay well hydrated — aim for 8–10 glasses of water per day.")

    # ── Exercise ──
    if risk_pct > 70:
        exercise.append("Start with supervised cardiac rehabilitation exercises.")
        exercise.append("Begin with 15–20 min of light walking; increase gradually.")
    elif risk_pct > 40:
        exercise.append("30 min of moderate aerobic activity (brisk walk, cycling) 5×/week.")
        exercise.append("Add resistance training 2×/week to improve insulin sensitivity.")
    else:
        exercise.append("150 min/week of moderate aerobic activity (WHO recommendation).")
        exercise.append("Include yoga or stretching for stress and BP reduction.")
    if v['bmi'] > 30:
        exercise.append("Low-impact activities (swimming, elliptical) to protect joints.")
    if v['smoking']:
        exercise.append("Join a smoking cessation program — smoking multiplies cardiac risk.")

    # ── Doctor ──
    if risk_pct > 70:
        doctor.append("🔴 HIGH PRIORITY: Consult a cardiologist within the next 7 days.")
        doctor.append("Request: Stress ECG, Echocardiogram, Full Lipid Panel, HbA1c.")
    elif risk_pct > 40:
        doctor.append("🟡 MODERATE: Schedule a cardiology review within 4 weeks.")
        doctor.append("Request: Fasting lipid profile, glucose tolerance test, 24-hr BP monitor.")
    else:
        doctor.append("🟢 LOW RISK: Annual preventive health check-up recommended.")
        doctor.append("Maintain current lifestyle and reassess in 12 months.")
    if v['diabetes_history'] or v['hba1c'] > 6.5:
        doctor.append("Endocrinologist consultation for diabetes management.")
    if v['family_history']:
        doctor.append("Inform cardiologist of family history for genetic risk screening.")

    return {'diet': diet, 'exercise': exercise, 'doctor': doctor}


## 10. PDF REPORT GENERATION


In [ ]:

def generate_pdf_report(
    input_values: dict,
    risk_pct: float,
    disease_risks: dict,
    explanations: list,
    recommendations: dict,
    shap_factors: list,
    output_path: str = "heart_disease_report.pdf"
):
    """Generate a detailed PDF report. Requires: pip install reportlab"""
    try:
        from reportlab.lib.pagesizes import A4
        from reportlab.lib import colors
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.units import cm
        from reportlab.platypus import (
            SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
        )
        from reportlab.lib.enums import TA_CENTER, TA_LEFT
    except ImportError:
        print("[PDF] reportlab not installed. Run: pip install reportlab")
        return None

    doc = SimpleDocTemplate(output_path, pagesize=A4,
                            rightMargin=2*cm, leftMargin=2*cm,
                            topMargin=2*cm, bottomMargin=2*cm)
    styles = getSampleStyleSheet()
    story = []

    RED   = colors.HexColor('#D32F2F')
    GREEN = colors.HexColor('#2E7D32')
    BLUE  = colors.HexColor('#1565C0')
    AMBER = colors.HexColor('#F57F17')

    risk_color = RED if risk_pct > 70 else (AMBER if risk_pct > 40 else GREEN)
    risk_label = "HIGH RISK" if risk_pct > 70 else ("MODERATE RISK" if risk_pct > 40 else "LOW RISK")

    title_style  = ParagraphStyle('Title',  parent=styles['Title'],  fontSize=20, textColor=BLUE, spaceAfter=6)
    h2_style     = ParagraphStyle('H2',     parent=styles['Heading2'], textColor=BLUE, spaceBefore=12, spaceAfter=4)
    body_style   = ParagraphStyle('Body',   parent=styles['Normal'],  fontSize=10, spaceAfter=3)
    risk_style   = ParagraphStyle('Risk',   parent=styles['Normal'],  fontSize=28,
                                  textColor=risk_color, alignment=TA_CENTER, spaceAfter=4)

    story.append(Paragraph("Heart Disease Risk Assessment Report", title_style))
    story.append(HRFlowable(width="100%", thickness=1, color=BLUE))
    story.append(Spacer(1, 0.3*cm))

    # Overall Risk
    story.append(Paragraph(f"Heart Disease Risk: {risk_pct}%", risk_style))
    story.append(Paragraph(f"Classification: {risk_label}", ParagraphStyle(
        'RL', parent=styles['Normal'], fontSize=14, textColor=risk_color, alignment=TA_CENTER)))
    story.append(Spacer(1, 0.5*cm))

    # Disease-Specific Risks
    story.append(Paragraph("Disease-Specific Risk Scores", h2_style))
    ds_data = [['Condition', 'Risk %']] + [[k, f"{v}%"] for k, v in disease_risks.items()]
    ds_table = Table(ds_data, colWidths=[10*cm, 4*cm])
    ds_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), BLUE),
        ('TEXTCOLOR',  (0,0), (-1,0), colors.white),
        ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
        ('GRID',       (0,0), (-1,-1), 0.5, colors.grey),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#EEF2F7')]),
        ('ALIGN', (1,0), (1,-1), 'CENTER'),
    ]))
    story.append(ds_table)
    story.append(Spacer(1, 0.4*cm))

    # Patient Parameters
    story.append(Paragraph("Patient Parameters", h2_style))
    param_rows = [['Parameter', 'Value', 'Description']]
    for feat in FEATURE_NAMES:
        param_rows.append([feat, str(input_values.get(feat, 'N/A')), FEATURE_DESCRIPTIONS[feat]])
    param_table = Table(param_rows, colWidths=[4*cm, 3*cm, 9*cm])
    param_table.setStyle(TableStyle([
        ('BACKGROUND',    (0,0), (-1,0), BLUE),
        ('TEXTCOLOR',     (0,0), (-1,0), colors.white),
        ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
        ('GRID',          (0,0), (-1,-1), 0.4, colors.grey),
        ('FONTSIZE',      (0,0), (-1,-1), 8),
        ('ROWBACKGROUNDS',(0,1), (-1,-1), [colors.white, colors.HexColor('#EEF2F7')]),
    ]))
    story.append(param_table)
    story.append(Spacer(1, 0.4*cm))

    # SHAP / Risk Explanations
    story.append(Paragraph("Key Risk Drivers", h2_style))
    for e in explanations:
        story.append(Paragraph(f"• {e}", body_style))
    story.append(Spacer(1, 0.4*cm))

    # Recommendations
    story.append(Paragraph("Health Recommendations", h2_style))
    story.append(Paragraph("<b>Diet</b>", body_style))
    for r in recommendations['diet']:
        story.append(Paragraph(f"  🥗 {r}", body_style))
    story.append(Paragraph("<b>Exercise</b>", body_style))
    for r in recommendations['exercise']:
        story.append(Paragraph(f"  🏃 {r}", body_style))
    story.append(Paragraph("<b>Medical Consultation</b>", body_style))
    for r in recommendations['doctor']:
        story.append(Paragraph(f"  🩺 {r}", body_style))

    story.append(Spacer(1, 0.5*cm))
    story.append(HRFlowable(width="100%", thickness=0.5, color=colors.grey))
    story.append(Paragraph(
        "<i>This report is generated by an AI model for informational purposes only. "
        "Always consult a qualified medical professional for diagnosis and treatment.</i>",
        ParagraphStyle('Disclaimer', parent=styles['Normal'], fontSize=8,
                       textColor=colors.grey, alignment=TA_CENTER)
    ))

    doc.build(story)
    print(f"[PDF] Report saved to: {output_path}")
    return output_path


## 11. OCR-BASED REPORT READING


In [ ]:

def extract_values_from_report(image_or_pdf_path: str, ocr_engine: str = "easyocr") -> dict:
    """
    Extract medical values from a scanned report image or PDF.

    Parameters:
        image_or_pdf_path : Path to image (.jpg/.png) or PDF file
        ocr_engine        : 'easyocr' or 'paddleocr'

    Returns:
        dict of extracted field values (keys matching FEATURE_NAMES where possible)
    """
    import re

    def _run_easyocr(path):
        import easyocr
        reader = easyocr.Reader(['en'], gpu=False)
        if path.lower().endswith('.pdf'):
            # Convert first page of PDF to image
            try:
                from pdf2image import convert_from_path
                imgs = convert_from_path(path, dpi=200, first_page=1, last_page=1)
                import tempfile, cv2, numpy as np
                tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
                imgs[0].save(tmp.name)
                results = reader.readtext(tmp.name)
            except ImportError:
                raise ImportError("Install pdf2image + poppler: pip install pdf2image")
        else:
            results = reader.readtext(path)
        return ' '.join([r[1] for r in results])

    def _run_paddleocr(path):
        from paddleocr import PaddleOCR
        ocr = PaddleOCR(use_angle_cls=True, lang='en')
        result = ocr.ocr(path, cls=True)
        texts = [line[1][0] for line in result[0]]
        return ' '.join(texts)

    # Run OCR
    print(f"[OCR] Extracting text from: {image_or_pdf_path} using {ocr_engine}")
    if ocr_engine == "easyocr":
        text = _run_easyocr(image_or_pdf_path)
    elif ocr_engine == "paddleocr":
        text = _run_paddleocr(image_or_pdf_path)
    else:
        raise ValueError("ocr_engine must be 'easyocr' or 'paddleocr'")

    print(f"[OCR] Raw text (first 300 chars): {text[:300]}")

    # ── Parse known fields from extracted text ──
    patterns = {
        'age':          r'\bage[\s:=]+(\d{1,3})',
        'trestbps':     r'(?:bp|blood pressure|systolic)[\s:=]+(\d{2,3})',
        'chol':         r'(?:cholesterol|total cholesterol)[\s:=]+(\d{2,3})',
        'ldl':          r'ldl[\s:=]+(\d{2,3})',
        'hdl':          r'hdl[\s:=]+(\d{2,3})',
        'triglycerides':r'(?:triglycerides|trig)[\s:=]+(\d{2,3})',
        'bmi':          r'bmi[\s:=]+(\d{1,2}\.?\d?)',
        'hba1c':        r'(?:hba1c|hba1c|a1c)[\s:=]+(\d{1,2}\.?\d?)',
        'fbs':          r'(?:fbs|fasting blood sugar)[\s:=]+(\d{2,3})',
        'thalach':      r'(?:max heart rate|thalach|hr max)[\s:=]+(\d{2,3})',
    }

    extracted = {}
    text_lower = text.lower()
    for field, pattern in patterns.items():
        m = re.search(pattern, text_lower)
        if m:
            extracted[field] = float(m.group(1))
            print(f"[OCR] Extracted {field} = {extracted[field]}")

    # Boolean fields
    extracted['smoking']          = int(bool(re.search(r'\bsmoker\b|\bsmoking\b', text_lower)))
    extracted['diabetes_history'] = int(bool(re.search(r'\bdiabetes\b|\bdiabetic\b', text_lower)))
    extracted['family_history']   = int(bool(re.search(r'family history', text_lower)))

    print(f"[OCR] Extracted {len(extracted)} fields.")
    return extracted


def autofill_inputs(extracted: dict, default_values: dict) -> dict:
    """Merge OCR-extracted values into a default inputs dict."""
    merged = default_values.copy()
    merged.update(extracted)
    return merged


## 12. MULTI-REPORT TYPE SUPPORT


In [ ]:

REPORT_FIELD_MAP = {
    "lipid_profile": {
        # Maps report field names → our feature names
        "total cholesterol": "chol",
        "ldl cholesterol":   "ldl",
        "hdl cholesterol":   "hdl",
        "triglycerides":     "triglycerides",
    },
    "ecg_report": {
        "heart rate":        "thalach",
        "st depression":     "oldpeak",
        "ecg result":        "restecg",
    },
    "echo_report": {
        "ejection fraction": None,   # not a direct feature but informative
        "wall motion":       None,
    },
    "complete_health_checkup": {
        # All fields supported
        "age": "age", "sex": "sex", "bmi": "bmi", "bp": "trestbps",
        "cholesterol": "chol", "ldl": "ldl", "hdl": "hdl",
        "triglycerides": "triglycerides", "hba1c": "hba1c",
        "fasting blood sugar": "fbs",
    }
}

def parse_report_by_type(text: str, report_type: str) -> dict:
    """Parse OCR text according to report type field mapping."""
    import re
    mapping = REPORT_FIELD_MAP.get(report_type, {})
    extracted = {}
    for report_field, feat_name in mapping.items():
        if feat_name is None:
            continue
        pattern = re.escape(report_field) + r'[\s:=]+(\d+\.?\d*)'
        m = re.search(pattern, text.lower())
        if m:
            extracted[feat_name] = float(m.group(1))
    return extracted


## 13. SAVE / LOAD MODEL


In [ ]:

def save_model(model, scaler, path_prefix="heart_disease_model"):
    with open(f"{path_prefix}.pkl", "wb") as f:
        pickle.dump({'model': model, 'scaler': scaler}, f)
    print(f"[Save] Model saved to {path_prefix}.pkl")


def load_model(path_prefix="heart_disease_model"):
    with open(f"{path_prefix}.pkl", "rb") as f:
        data = pickle.load(f)
    print(f"[Load] Model loaded from {path_prefix}.pkl")
    return data['model'], data['scaler']


## 14. FULL PIPELINE — SINGLE PREDICTION


In [ ]:

def full_prediction_pipeline(
    model,
    scaler,
    X_train_raw: pd.DataFrame,
    input_values: dict,
    generate_pdf: bool = True,
    pdf_path: str = "report.pdf"
):
    """Run the complete prediction pipeline and print a formatted report."""

    print("\n" + "="*60)
    print("   HEART DISEASE RISK ASSESSMENT")
    print("="*60)

    # ── Risk % ──
    risk_pct = predict_risk_percentage(model, scaler, input_values)
    risk_label = "HIGH" if risk_pct > 70 else ("MODERATE" if risk_pct > 40 else "LOW")
    print(f"\n  Heart Disease Risk = {risk_pct}%  [{risk_label} RISK]")

    # ── Disease-Specific ──
    disease_risks = disease_specific_risks(input_values, risk_pct)
    print("\n  Disease-Specific Risk Scores:")
    for disease, pct in disease_risks.items():
        bar = "█" * int(pct / 5)
        print(f"    {disease:<30} {pct:>5}%  {bar}")

    # ── SHAP Explanation ──
    shap_factors = explain_with_shap(model, scaler, X_train_raw, input_values)

    # ── Risk Explanations ──
    explanations = explain_predictions(input_values, shap_factors)
    print("\n  Key Risk Factors:")
    for e in explanations:
        print(f"    • {e}")

    # ── SHAP Summary ──
    if shap_factors:
        print("\n  SHAP Feature Contributions (top 5):")
        for feat, pct_contrib, direction, _ in shap_factors[:5]:
            print(f"    {FEATURE_DESCRIPTIONS[feat]:<40} {direction}  +{pct_contrib}% risk")

    # ── Recommendations ──
    recs = generate_recommendations(input_values, risk_pct)
    print("\n  Health Recommendations:")
    print("  [Diet]")
    for r in recs['diet'][:3]:
        print(f"    🥗 {r}")
    print("  [Exercise]")
    for r in recs['exercise'][:2]:
        print(f"    🏃 {r}")
    print("  [Doctor]")
    for r in recs['doctor']:
        print(f"    🩺 {r}")

    # ── PDF ──
    if generate_pdf:
        generate_pdf_report(
            input_values, risk_pct, disease_risks,
            explanations, recs, shap_factors, output_path=pdf_path
        )

    print("\n" + "="*60)
    return {
        'risk_pct': risk_pct,
        'risk_label': risk_label,
        'disease_risks': disease_risks,
        'explanations': explanations,
        'recommendations': recs,
        'shap_factors': shap_factors
    }


## 15. MAIN — Train, evaluate, demo prediction


In [ ]:

if __name__ == "__main__":
    CSV_PATH = "heart_disease_data_enhanced.csv"

    # ── Train ──
    df = load_data(CSV_PATH)
    X_train_s, X_test_s, y_train, y_test, scaler, X_train_raw, X_test_raw = preprocess(df)
    model = train_model(X_train_s, y_train, X_test_s, y_test)
    save_model(model, scaler)

    # ── Sample prediction ──
    sample_patient = {
        # Original 13 features
        'age': 58,
        'sex': 1,
        'cp': 0,            # typical angina
        'trestbps': 150,
        'chol': 280,
        'fbs': 1,
        'restecg': 1,
        'thalach': 120,
        'exang': 1,
        'oldpeak': 3.5,
        'slope': 0,
        'ca': 2,
        'thal': 2,
        # New 8 features
        'bmi': 31.2,
        'hdl': 32,
        'ldl': 195,
        'triglycerides': 310,
        'smoking': 1,
        'family_history': 1,
        'diabetes_history': 1,
        'hba1c': 7.8,
    }

    results = full_prediction_pipeline(
        model, scaler, X_train_raw, sample_patient,
        generate_pdf=True, pdf_path="sample_report.pdf"
    )

    # ── OCR Demo (commented out — requires EasyOCR/PaddleOCR and an image) ──
    # extracted = extract_values_from_report("lab_report.jpg", ocr_engine="easyocr")
    # filled = autofill_inputs(extracted, sample_patient)
    # full_prediction_pipeline(model, scaler, X_train_raw, filled)
